## Encryption Example 

In [ ]:
from o6 import Client, StatusCodeError, types, Server, SecurityMode, SecurityPolicy

### Encrypted Server

In [ ]:
from o6.crypto import create_self_signed_certificate

srv_key, srv_cert = create_self_signed_certificate(
        app_uri="urn:example:server",
        common_name="ExampleServer@localhost",
    )

cli_key, cli_cert = create_self_signed_certificate(
        app_uri="urn:example:client",
        common_name="ExampleClient@localhost",
    )

In [ ]:
s = Server(port=4840,
        certificate=srv_cert,
        private_key=srv_key,
        trust_list=[cli_cert],
        accept_all_certificates=True,
        application_uri="urn:example:server",)
s.start()
c = Client("opc.tcp://localhost:4840", 
           certificate=cli_cert,
           private_key=cli_key, 
           trust_list=[srv_cert],
           security_mode=SecurityMode.SIGN_AND_ENCRYPT,
           security_policy=SecurityPolicy.BASIC256SHA256,
           application_uri="urn:example:client",)
s.add_variable("Temperature", s.objects_node, 15.0, nodeid="ns=1;s=Temperature")
await c.connect()

print(await c.read("ns=1;s=Temperature"))

In [ ]:

await c.disconnect()
s.stop()